# AUTokens50 Chonkie Chunking Exploration

Objective:
- Respond to Matthew Altenburg's email from Monday, 2026-08-31: stop extending SimHash/MinHash reduction work and focus this week on understanding chunking with the open source Chonkie library.
- Build a small, reviewable chunking workflow for the existing `AUTokens50_with_hash_simhash` parquet-part dataset.
- Keep source data read-only and write only preview/audit outputs to Scratch or another explicit output directory.

Success criteria:
- The notebook can run on the online Jupyter environment that contains `JoeyLLM_Data/AUTokens50_with_hash_simhash/part_*.parquet`.
- It samples a bounded number of rows from the `text` column, chunks them with Chonkie, and records chunk-level metadata for later vector-database preparation.
- If full data is unavailable, the sample fallback still demonstrates the chunking process clearly enough for Matthew to review.


## Plan

1. Check/install minimal dependencies: `chonkie`, `pyarrow`, and `pandas`.
2. Resolve the immutable source dataset from common Jupyter paths, including `JoeyLLM_Data/AUTokens50_with_hash_simhash` from the screenshot.
3. Load only a small bounded sample from `part_*.parquet` files so this does not accidentally process many gigabytes.
4. Compare Chonkie's `TokenChunker`, `SentenceChunker`, and `RecursiveChunker` on the same sample rows.
5. Save preview chunks and a compact JSON summary outside the source directory.
6. Use the results to decide the next production chunking strategy.

Important: this notebook is a chunking preview, not a production vector database integration.


## Cell 1 - Dependency Check

Chonkie documentation says the base install includes local `TokenChunker`, `SentenceChunker`, and `RecursiveChunker`. This cell installs only missing packages inside the notebook kernel.


In [1]:
# Purpose: Install only missing runtime dependencies for the chunking preview.
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "chonkie": "chonkie",
    "pyarrow": "pyarrow",
    "pandas": "pandas",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already available.")


Installing missing packages: ['chonkie']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [chonkie]m3/4 [chonkie]


## Cell 2 - Imports And Configuration

The dataset path is intentionally resolved from a list of candidates. The source folder in the screenshot is read-only, so outputs go to Scratch by default.


In [2]:
# Purpose: Define paths, sampling limits, chunker settings, and output location.
from pathlib import Path
import json
import math
import statistics
import time
from typing import Iterable

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from chonkie import RecursiveChunker, SentenceChunker, TokenChunker

TEXT_COL = "text"
HASH_COL = "hash"
SIMHASH_COL = "simhash"

# Keep the first run intentionally small. Increase only after reviewing preview quality.
MAX_FILES = 2
MAX_ROWS_PER_FILE = 25
MAX_DOCS_TO_CHUNK = 20
PREVIEW_TEXT_CHARS = 220

CHUNK_SIZE = 2048
CHUNK_OVERLAP = 128
TOKENIZER = "character"

INPUT_DIR_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None

INPUT_CANDIDATES = [
    Path.cwd() / "JoeyLLM_Data/AUTokens50_with_hash_simhash",
    Path("/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/notebook/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/data/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/JoeyLLM_data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/notebook/JoeyLLM_data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/data/JoeyLLM_data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash"),
]

SCRATCH_CANDIDATES = [
    Path.cwd() / "Scratch",
    Path("/Scratch"),
    Path("/home/jovyan/Scratch"),
    Path("/home/jovyan/notebook/Scratch"),
    Path("/scratch"),
    Path.cwd() / "chunking_output",
]

SAMPLE_TEXTS = [
    "JoeyLLM needs reliable chunking before embeddings and vector database loading. This preview uses Chonkie locally and records chunk metadata for review.",
    "Exact hash deduplication was validated last week. The next step is to split cleaned documents into bounded, useful pieces for retrieval augmented generation.",
    "A good chunk should preserve enough context to answer a question while staying small enough for embedding and retrieval efficiency.",
]

print({
    "max_files": MAX_FILES,
    "max_rows_per_file": MAX_ROWS_PER_FILE,
    "max_docs_to_chunk": MAX_DOCS_TO_CHUNK,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "tokenizer": TOKENIZER,
})


{'max_files': 2, 'max_rows_per_file': 25, 'max_docs_to_chunk': 20, 'chunk_size': 2048, 'chunk_overlap': 128, 'tokenizer': 'character'}


## Cell 3 - Resolve Dataset And Output Paths

This cell does not write to the source. It only discovers `part_*.parquet` files and prepares an output folder.


In [3]:
# Purpose: Resolve source parquet files and notebook-owned output paths.
def part_number(path: Path) -> int:
    try:
        return int(path.stem.split("_")[-1])
    except ValueError:
        return 10**12


def first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


input_dir = Path(INPUT_DIR_OVERRIDE).expanduser().resolve() if INPUT_DIR_OVERRIDE else first_existing(INPUT_CANDIDATES)
scratch_base = first_existing(SCRATCH_CANDIDATES) or (Path.cwd() / "chunking_output")
output_dir = Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve() if OUTPUT_DIR_OVERRIDE else (scratch_base / "AUTokens50_chonkie_chunk_preview")
output_dir.mkdir(parents=True, exist_ok=True)

if input_dir is None:
    parquet_files = []
    print("No AUTokens50 parquet directory found in this environment. The notebook will use SAMPLE_TEXTS fallback.")
else:
    parquet_files = sorted(input_dir.glob("part_*.parquet"), key=part_number)
    print(f"Input directory: {input_dir}")
    print(f"Parquet part files found: {len(parquet_files)}")

print(f"Output directory: {output_dir}")
print("First files:", [p.name for p in parquet_files[:5]])


Input directory: /home/jovyan/JoeyLLM_Data/AUTokens50_with_hash_simhash
Parquet part files found: 57
Output directory: /home/jovyan/Scratch/AUTokens50_chonkie_chunk_preview
First files: ['part_0.parquet', 'part_1.parquet', 'part_2.parquet', 'part_3.parquet', 'part_4.parquet']


## Cell 4 - Inspect Schema And Load Bounded Sample

The screenshot shows large files, for example `part_9.parquet` around 2.4 GB. This cell reads only selected columns and only a small number of rows per file.


In [4]:
# Purpose: Load a tiny, reviewable sample without scanning the full dataset.
def read_sample_from_file(path: Path, max_rows: int) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    schema_names = pf.schema_arrow.names
    if TEXT_COL not in schema_names:
        raise ValueError(f"{path} is missing required text column: {TEXT_COL}")

    columns = [col for col in [TEXT_COL, HASH_COL, SIMHASH_COL] if col in schema_names]
    batches = []
    rows_remaining = max_rows
    for batch in pf.iter_batches(batch_size=min(max_rows, 1024), columns=columns):
        table = pa.Table.from_batches([batch])
        if table.num_rows > rows_remaining:
            table = table.slice(0, rows_remaining)
        batches.append(table)
        rows_remaining -= table.num_rows
        if rows_remaining <= 0:
            break

    if not batches:
        return pd.DataFrame(columns=columns)
    df = pa.concat_tables(batches).to_pandas()
    df.insert(0, "source_file", path.name)
    return df


sample_frames = []
for path in parquet_files[:MAX_FILES]:
    sample_frames.append(read_sample_from_file(path, MAX_ROWS_PER_FILE))

if sample_frames:
    docs_df = pd.concat(sample_frames, ignore_index=True)
    docs_df = docs_df[docs_df[TEXT_COL].notna()].copy().head(MAX_DOCS_TO_CHUNK)
    docs_df[TEXT_COL] = docs_df[TEXT_COL].astype(str)
    data_mode = "parquet_sample"
else:
    docs_df = pd.DataFrame({
        "source_file": ["fallback_sample"] * len(SAMPLE_TEXTS),
        TEXT_COL: SAMPLE_TEXTS,
        HASH_COL: [None] * len(SAMPLE_TEXTS),
        SIMHASH_COL: [None] * len(SAMPLE_TEXTS),
    })
    data_mode = "fallback_sample"

print("Data mode:", data_mode)
print("Documents loaded:", len(docs_df))
docs_df[["source_file", TEXT_COL]].assign(text_preview=lambda df: df[TEXT_COL].str.slice(0, PREVIEW_TEXT_CHARS))[["source_file", "text_preview"]].head(5)


Data mode: parquet_sample
Documents loaded: 20


,source_file,text_preview
0,part_0.parquet,"Justine Davies –, Monday, January, 31, 2011, (..."
1,part_0.parquet,Interview: Jenny Macpherson\n- by Rowena Scott...
2,part_0.parquet,The foundations for successful riding\n19 post...
3,part_0.parquet,photos by Carlo Ledesma\nWhen God was creating...
4,part_0.parquet,Joanne Harris is apparently as formidable a Yo...


## Cell 5 - Configure Chonkie Chunkers

The first pass uses character tokenization so it can run without downloading external tokenizer assets. We compare three strategies on the same rows:

- `TokenChunker`: fixed-size chunks with overlap.
- `SentenceChunker`: preserves sentence boundaries where possible.
- `RecursiveChunker`: recursively splits structured/long text.


In [5]:
# Purpose: Create comparable Chonkie chunkers for the preview.
chunkers = {
    "token": TokenChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
    "sentence": SentenceChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, min_sentences_per_chunk=1),
    "recursive": RecursiveChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE),
}

chunkers


{'token': TokenChunker(tokenizer=<chonkie.tokenizer.ChonkieAutoTokenizer object at 0x7912554d0590>, chunk_size=2048, chunk_overlap=128),
 'sentence': SentenceChunker(tokenizer=<chonkie.tokenizer.ChonkieAutoTokenizer object at 0x7912556c1310>, chunk_size=2048, chunk_overlap=128, min_sentences_per_chunk=1, min_characters_per_sentence=12, approximate=False, delim=['. ', '! ', '? ', '\n'], include_delim=prev),
 'recursive': RecursiveChunker(tokenizer=<chonkie.tokenizer.ChonkieAutoTokenizer object at 0x7912556c2210>, rules=RecursiveRules(levels=[RecursiveLevel(delimiters=['\n\n', '\r\n', '\n', '\r'], whitespace=False, include_delim=prev, pattern=None, pattern_mode=split), RecursiveLevel(delimiters=['. ', '! ', '? '], whitespace=False, include_delim=prev, pattern=None, pattern_mode=split), RecursiveLevel(delimiters=['{', '}', '"', '[', ']', '<', '>', '(', ')', ':', ';', ',', '—', '|', '~', '-', '...', '`', "'"], whitespace=False, include_delim=prev, pattern=None, pattern_mode=split), Recursi

## Cell 6 - Run Chunking Preview

Each output chunk keeps provenance: original document index, source part file, original hash/simhash if present, chunk index, character offsets if Chonkie provides them, and token count.


In [6]:
# Purpose: Chunk the sampled documents and normalize Chonkie Chunk objects into rows.
def normalize_chunk(chunk, fallback_index: int) -> dict:
    text = getattr(chunk, "text", str(chunk))
    return {
        "chunk_text": text,
        "chunk_token_count": getattr(chunk, "token_count", None),
        "chunk_start_index": getattr(chunk, "start_index", None),
        "chunk_end_index": getattr(chunk, "end_index", None),
        "chunk_char_count": len(text),
        "chunk_preview": text[:PREVIEW_TEXT_CHARS],
    }


chunk_rows = []
started = time.time()
for doc_index, row in docs_df.reset_index(drop=True).iterrows():
    text = row[TEXT_COL]
    for chunker_name, chunker in chunkers.items():
        chunks = chunker.chunk(text)
        for chunk_index, chunk in enumerate(chunks):
            item = normalize_chunk(chunk, chunk_index)
            item.update({
                "chunker": chunker_name,
                "doc_index": int(doc_index),
                "source_file": row.get("source_file"),
                "source_hash": row.get(HASH_COL),
                "source_simhash": row.get(SIMHASH_COL),
                "chunk_index": int(chunk_index),
                "source_text_chars": len(text),
            })
            chunk_rows.append(item)

chunks_df = pd.DataFrame(chunk_rows)
elapsed_seconds = round(time.time() - started, 3)
print(f"Created {len(chunks_df)} preview chunks from {len(docs_df)} documents in {elapsed_seconds}s")
chunks_df.head(10)


Created 328 preview chunks from 20 documents in 0.047s


,chunk_text,chunk_token_count,chunk_start_index,chunk_end_index,chunk_char_count,chunk_preview,chunker,doc_index,source_file,source_hash,source_simhash,chunk_index,source_text_chars
0,"Justine Davies –, Monday, January, 31, 2011, (...",2048,0,2048,2048,"Justine Davies –, Monday, January, 31, 2011, (...",token,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,0,6095
1,?\nContinue reading 'What is the best small bu...,2048,1920,3968,2048,?\nContinue reading 'What is the best small bu...,token,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,1,6095
2,inue reading 'We have Medicare – should we hav...,2048,3840,5888,2048,inue reading 'We have Medicare – should we hav...,token,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,2,6095
3,2011\n- May 2011\n- April 2011\n- March 2011\n...,335,5760,6095,335,2011\n- May 2011\n- April 2011\n- March 2011\n...,token,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,3,6095
4,"Justine Davies –, Monday, January, 31, 2011, (...",2024,0,2024,2024,"Justine Davies –, Monday, January, 31, 2011, (...",sentence,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,0,6095
5,Continue reading 'What is the best small busin...,2018,1922,3940,2018,Continue reading 'What is the best small busin...,sentence,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,1,6095
6,Continue reading 'We have Medicare – should we...,2044,3836,5880,2044,Continue reading 'We have Medicare – should we...,sentence,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,2,6095
7,- April 2011\n- March 2011\n- February 2011\n-...,319,5776,6095,319,- April 2011\n- March 2011\n- February 2011\n-...,sentence,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,3,6095
8,"Justine Davies –, Monday, January, 31, 2011, (...",2024,0,2024,2024,"Justine Davies –, Monday, January, 31, 2011, (...",recursive,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,0,6095
9,"Justine Davies –, Wednesday, January, 19, 2011...",1970,2024,3994,1970,"Justine Davies –, Wednesday, January, 19, 2011...",recursive,0,part_0.parquet,ea089171825fc995ed73549096bca65f42cc8b70bbea88...,3320040830886190886,1,6095


## Cell 7 - Compare Chunker Metrics

These metrics are for understanding behavior, not final production acceptance. Review chunk quality manually before scaling up.


In [7]:
# Purpose: Summarize chunk counts and size distributions by chunker.
def safe_mean(values: pd.Series) -> float | None:
    values = values.dropna()
    if len(values) == 0:
        return None
    return round(float(values.mean()), 2)


summary_df = (
    chunks_df.groupby("chunker")
    .agg(
        documents=("doc_index", "nunique"),
        chunks=("chunk_index", "count"),
        avg_chunks_per_doc=("doc_index", lambda s: round(len(s) / max(1, s.nunique()), 2)),
        min_chunk_chars=("chunk_char_count", "min"),
        avg_chunk_chars=("chunk_char_count", lambda s: round(float(s.mean()), 2)),
        max_chunk_chars=("chunk_char_count", "max"),
        avg_token_count=("chunk_token_count", safe_mean),
    )
    .reset_index()
)
summary_df


,chunker,documents,chunks,avg_chunks_per_doc,min_chunk_chars,avg_chunk_chars,max_chunk_chars,avg_token_count
0,recursive,20,112,5.6,152,1657.29,2045,1657.29
1,sentence,20,108,5.4,134,1768.31,2048,1768.31
2,token,20,108,5.4,186,1822.97,2048,1822.97


## Cell 8 - Save Preview Outputs

The saved files are deliberately small and reviewable. They can be attached or shown to Matthew without needing to share the complete dataset.


In [8]:
# Purpose: Persist small preview artifacts outside the immutable source directory.
preview_parquet = output_dir / "chonkie_chunk_preview.parquet"
preview_csv = output_dir / "chonkie_chunk_preview.csv"
summary_json = output_dir / "chonkie_chunk_preview_summary.json"

chunks_df.to_parquet(preview_parquet, index=False)
chunks_df.drop(columns=["chunk_text"], errors="ignore").to_csv(preview_csv, index=False)

summary = {
    "task": "AUTokens50 Chonkie chunking preview",
    "email_date": "2026-08-31",
    "data_mode": data_mode,
    "input_dir": str(input_dir) if input_dir else None,
    "output_dir": str(output_dir),
    "files_seen": len(parquet_files),
    "files_sampled": min(MAX_FILES, len(parquet_files)),
    "documents_sampled": int(len(docs_df)),
    "chunkers": list(chunkers.keys()),
    "tokenizer": TOKENIZER,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "preview_chunks": int(len(chunks_df)),
    "elapsed_seconds": elapsed_seconds,
    "metrics": summary_df.to_dict(orient="records"),
    "outputs": {
        "preview_parquet": str(preview_parquet),
        "preview_csv": str(preview_csv),
        "summary_json": str(summary_json),
    },
    "notes": [
        "Source parquet files were treated as read-only.",
        "This is a bounded preview for understanding chunking, not a full production run.",
        "Use the cleaned exact-hash deduped dataset as input if available; otherwise use the original hash/simhash dataset for exploration only.",
    ],
}
summary_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "task": "AUTokens50 Chonkie chunking preview",
  "email_date": "2026-08-31",
  "data_mode": "parquet_sample",
  "input_dir": "/home/jovyan/JoeyLLM_Data/AUTokens50_with_hash_simhash",
  "output_dir": "/home/jovyan/Scratch/AUTokens50_chonkie_chunk_preview",
  "files_seen": 57,
  "files_sampled": 2,
  "documents_sampled": 20,
  "chunkers": [
    "token",
    "sentence",
    "recursive"
  ],
  "tokenizer": "character",
  "chunk_size": 2048,
  "chunk_overlap": 128,
  "preview_chunks": 328,
  "elapsed_seconds": 0.047,
  "metrics": [
    {
      "chunker": "recursive",
      "documents": 20,
      "chunks": 112,
      "avg_chunks_per_doc": 5.6,
      "min_chunk_chars": 152,
      "avg_chunk_chars": 1657.29,
      "max_chunk_chars": 2045,
      "avg_token_count": 1657.29
    },
    {
      "chunker": "sentence",
      "documents": 20,
      "chunks": 108,
      "avg_chunks_per_doc": 5.4,
      "min_chunk_chars": 134,
      "avg_chunk_chars": 1768.31,
      "max_chunk_chars": 2048,
      "a

## Cell 9 - Manual Review Samples

Use this cell to inspect whether chunks look useful for retrieval. Good chunks should preserve answerable context without becoming too large.


In [9]:
# Purpose: Show compact examples for human review.
for chunker_name in chunkers:
    print("\n" + "=" * 80)
    print(f"Chunker: {chunker_name}")
    examples = chunks_df[chunks_df["chunker"] == chunker_name].head(3)
    for _, item in examples.iterrows():
        print("-" * 80)
        print({
            "doc_index": int(item["doc_index"]),
            "chunk_index": int(item["chunk_index"]),
            "chars": int(item["chunk_char_count"]),
            "tokens": None if pd.isna(item["chunk_token_count"]) else int(item["chunk_token_count"]),
        })
        print(item["chunk_preview"])



Chunker: token
--------------------------------------------------------------------------------
{'doc_index': 0, 'chunk_index': 0, 'chars': 2048, 'tokens': 2048}
Justine Davies –, Monday, January, 31, 2011, (10:09pm)
Well, not necessarily just Gen Y – but there was an amusing article in the Sunday Mail on the weekend, titled ‘Gen Y Women losing “Female” Skills’. The article was c
--------------------------------------------------------------------------------
{'doc_index': 0, 'chunk_index': 1, 'chars': 2048, 'tokens': 2048}
?
Continue reading 'What is the best small business you’ve shopped at, ever?'
|41 comments | Permalink|
Justine Davies –, Wednesday, January, 19, 2011, (2:41pm)
I’ve just finished contacting books (disgusting, sticky stu
--------------------------------------------------------------------------------
{'doc_index': 0, 'chunk_index': 2, 'chars': 2048, 'tokens': 2048}
inue reading 'We have Medicare – should we have “Disastercare” as well?'
|227 comments | Permalink|
J

## Decision Notes

Initial decision for this week:
- Chonkie is appropriate for the chunking exploration because it offers local open-source chunkers and does not require vector database infrastructure to start.
- Start with bounded preview rows and character tokenizer for reliability in the online Jupyter environment.
- Prefer `RecursiveChunker` or `SentenceChunker` if manual review shows better semantic boundaries than fixed token chunking.
- Do not claim production integration until the full deduped dataset is chunked, validated, and connected to the embedding/vector pipeline.

Next steps:
- Run this notebook online against `JoeyLLM_Data/AUTokens50_with_hash_simhash/part_*.parquet` or the exact-hash deduped output if Matthew wants the cleaned corpus.
- Review `chonkie_chunk_preview.csv` and sample text chunks.
- After Matthew confirms chunk size and strategy, scale from preview sampling to all parts in a separate production notebook/script.


## Final Notes - Chonkie Chunking Preview

This notebook responds to Matthew's 2026-08-31 request to shift this week's focus from SimHash/MinHash reduction to chunking with the open source Chonkie library.

The preview run completed successfully on the online Jupyter environment:

- Input dataset found: `/home/jovyan/JoeyLLM_Data/AUTokens50_with_hash_simhash`
- Parquet part files found: `57`
- Sampled files: `2`
- Sampled documents: `20`
- Chunkers tested: `TokenChunker`, `SentenceChunker`, `RecursiveChunker`
- Preview chunks created: `328`
- Output directory: `/home/jovyan/Scratch/AUTokens50_chonkie_chunk_preview`

The chunker comparison from this bounded preview was:

| Chunker | Documents | Chunks | Average Chunks per Document | Average Chunk Characters |
|---|---:|---:|---:|---:|
| RecursiveChunker | 20 | 112 | 5.6 | 1657.29 |
| SentenceChunker | 20 | 108 | 5.4 | 1768.31 |
| TokenChunker | 20 | 108 | 5.4 | 1822.97 |

Initial observation:

- All three Chonkie chunkers ran successfully on sampled AUTokens50 text.
- `TokenChunker` creates fixed-size chunks and can split in the middle of words or sentences.
- `SentenceChunker` and `RecursiveChunker` appear more useful for review because they better preserve text boundaries.
- `RecursiveChunker` produced slightly more chunks with a smaller average chunk size, which may be useful for later retrieval and vector database preparation.

This notebook should be treated as a chunking exploration and preview only. It does not claim that the full corpus has been chunked, and it does not implement embeddings or vector database integration yet.

Recommended next step:

Use this preview to confirm the preferred chunking strategy and chunk size, then run a separate full-corpus chunking notebook/script against the deduplicated dataset or the approved source dataset.